# Wind Power Reliability Analysis: UK Reliable Capacity Recommendation

## Objective
Analyze historical actual wind power generation data to understand reliability characteristics and recommend how many MW of wind power capacity can be reliably expected to meet electricity demand.

## Key Question
**How many MW of wind power can utilities reliably depend on to meet electricity demand?**

## Methodology
1. Fetch historical actual wind generation (FUELHH) from Jan 2025 onwards
2. Analyze generation distribution and statistical characteristics
3. Calculate capacity factors and percentile outputs
4. Analyze weather-dependent reliability patterns
5. Identify peak demand periods (winter evenings) and wind availability
6. Recommend reliable capacity with supporting evidence

## Important Assumptions
- "Reliable" implies high-confidence availability for grid operations
- Analysis focuses on capacity credit (effective capacity for meeting peak demand)
- Weather patterns and seasonal variations considered
- Data resolution: 30-minute intervals (half-hourly)

# Notebook 2: Reliable Wind Power Capacity Analysis

**Question**: Based on historical wind generation data, **how many MW of wind power can we reliably expect to be available to meet electricity demand?**

## Framing the Problem

This is fundamentally a question about **capacity credit** — the fraction of installed wind capacity that can be relied upon to meet demand, especially during critical periods (peak demand, cold dark doldrums).

**Key trade-off**: 
- Quoting a high number (e.g. average output ~8–10 GW) is too optimistic — there will be periods when wind is far below this.
- Quoting too low a number (e.g. minimum ever recorded) is too conservative — it would undervalue wind's real contribution.

**Approach**: We will use **percentile analysis** to provide a range of reliability estimates:
- **P10 (10th percentile)**: Wind output exceeded 90% of the time — a conservative, high-reliability figure.
- **P50 (median)**: Exceeded 50% of the time — a central estimate.
- **P90 (90th percentile)**: Exceeded 10% of the time — optimistic, expected average conditions.

We also perform **stress testing**: what does wind output look like during periods of peak electricity demand (winter evenings)?

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#64748b',
    'ytick.color': '#64748b',
    'text.color': '#e2e8f0',
    'grid.color': '#334155',
    'grid.linewidth': 0.5,
    'axes.titlecolor': '#f1f5f9',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 120,
})

BLUE   = '#3b82f6'
GREEN  = '#10b981'
AMBER  = '#f59e0b'
RED    = '#ef4444'
PURPLE = '#a78bfa'
TEAL   = '#14b8a6'

print('Ready.')

## 1. Fetch Historical Actual Wind Generation

In [ ]:
BASE = 'https://data.elexon.co.uk/bmrs/api/v1'

def fetch_actuals_range(date_from: str, date_to: str) -> pd.DataFrame:
    url = (
        f"{BASE}/datasets/FUELHH/stream"
        f"?settlementDateFrom={date_from}"
        f"&settlementDateTo={date_to}"
        f"&fuelType=WIND"
    )
    print(f'Fetching: {date_from} → {date_to}', end=' ... ')
    resp = requests.get(url, headers={'Accept': 'application/json'}, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    rows = data if isinstance(data, list) else data.get('data', data.get('items', []))

    df = pd.DataFrame(rows)
    df = df[df['fuelType'] == 'WIND'].copy()
    df['target_time'] = pd.to_datetime(df['startTime'], utc=True)
    df['actual_mw'] = pd.to_numeric(df['generation'], errors='coerce')
    df = df[['target_time', 'actual_mw']].dropna()
    df = df.groupby('target_time', as_index=False)['actual_mw'].max()
    print(f'{len(df):,} records')
    return df


# Fetch Jan–Mar 2025 (extend to more months if API allows)
actuals = fetch_actuals_range('2025-01-01T00:00:00', '2025-03-31T23:59:59')

# Feature engineering
actuals['date']        = actuals['target_time'].dt.date
actuals['month']       = actuals['target_time'].dt.month
actuals['month_name']  = actuals['target_time'].dt.strftime('%b %Y')
actuals['hour']        = actuals['target_time'].dt.hour
actuals['weekday']     = actuals['target_time'].dt.dayofweek  # 0=Mon
actuals['week']        = actuals['target_time'].dt.isocalendar().week.astype(int)

# Peak demand period: winter weekdays 16:00–20:00 UTC
actuals['is_peak'] = (
    (actuals['hour'] >= 16) & (actuals['hour'] < 20) &
    (actuals['weekday'] < 5) &
    (actuals['month'].isin([1, 2, 3, 10, 11, 12]))
)

print(f'\nDataset: {len(actuals):,} half-hourly records')
print(f'Range: {actuals.target_time.min()} → {actuals.target_time.max()}')
print(f'Generation range: {actuals.actual_mw.min():.0f}–{actuals.actual_mw.max():.0f} MW')
print(f'Mean: {actuals.actual_mw.mean():.0f} MW | Median: {actuals.actual_mw.median():.0f} MW')

## 2. Overall Distribution of Wind Generation

In [ ]:
pcts = [5, 10, 25, 50, 75, 90, 95]
percentile_vals = {p: np.percentile(actuals['actual_mw'], p) for p in pcts}

print('Generation Percentiles:')
for p, v in percentile_vals.items():
    print(f'  P{p:>2}: {v:>7,.0f} MW')
print(f'  Min: {actuals.actual_mw.min():>7,.0f} MW')
print(f'  Max: {actuals.actual_mw.max():>7,.0f} MW')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('UK Wind Generation Distribution (Jan–Mar 2025)', color='#f1f5f9', fontsize=14)

# Histogram
ax = axes[0]
ax.hist(actuals['actual_mw'], bins=80, color=BLUE, alpha=0.8, edgecolor='none')
colors_p = {10: RED, 50: AMBER, 90: GREEN}
for p, col in colors_p.items():
    v = percentile_vals[p]
    ax.axvline(v, color=col, linewidth=1.5, linestyle='--')
    ax.text(v + 50, ax.get_ylim()[1] * 0.85, f'P{p}\n{v:.0f}MW', color=col, fontsize=8)
ax.set_xlabel('Wind Generation (MW)')
ax.set_ylabel('Frequency (30-min periods)')
ax.set_title('Generation Distribution')
ax.set_facecolor('#1e293b')

# Exceedance curve (reliability curve)
ax = axes[1]
sorted_gen = np.sort(actuals['actual_mw'].values)[::-1]  # descending
exceedance = np.arange(1, len(sorted_gen) + 1) / len(sorted_gen) * 100  # % of time
ax.plot(exceedance, sorted_gen, color=GREEN, linewidth=2)
for p, col in colors_p.items():
    v = percentile_vals[p]
    exc = 100 - p  # exceedance = 100 - percentile
    ax.plot(exc, v, 'o', color=col, markersize=7, zorder=5)
    ax.annotate(f'P{p} = {v:.0f}MW\n({exc}% of time)', xy=(exc, v),
                xytext=(exc + 3, v + 200), color=col, fontsize=8,
                arrowprops=dict(arrowstyle='->', color=col, lw=0.8))
ax.set_xlabel('% of time wind exceeds this level')
ax.set_ylabel('Wind Generation (MW)')
ax.set_title('Reliability / Exceedance Curve')
ax.set_xlim(0, 100)
ax.set_facecolor('#1e293b')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wind_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Wind Generation During Peak Demand Periods

**Motivation**: The grid operator needs to ensure supply meets demand during **peak demand** (winter weekday evenings, ~16:00–20:00 UTC). This is when reliable wind output matters most.

Wind output during peak periods may differ from the overall distribution — this is the "capacity credit" question.

In [ ]:
peak = actuals[actuals['is_peak']]
offpeak = actuals[~actuals['is_peak']]

print(f'Peak demand periods: {len(peak):,} half-hourly slots')
print(f'Off-peak periods:    {len(offpeak):,} half-hourly slots')

peak_pcts   = {p: np.percentile(peak['actual_mw'], p) for p in pcts}
offpk_pcts  = {p: np.percentile(offpeak['actual_mw'], p) for p in pcts}

print('\nPercentile comparison: Peak vs Off-peak')
print(f'{"Pct":<8} {"Peak (MW)":>12} {"Off-peak (MW)":>14}')
for p in pcts:
    print(f'P{p:<7} {peak_pcts[p]:>12,.0f} {offpk_pcts[p]:>14,.0f}')

fig, ax = plt.subplots(figsize=(12, 5))
sorted_peak = np.sort(peak['actual_mw'].values)[::-1]
sorted_offpk = np.sort(offpeak['actual_mw'].values)[::-1]
exc_pk = np.arange(1, len(sorted_peak)+1) / len(sorted_peak) * 100
exc_op = np.arange(1, len(sorted_offpk)+1) / len(sorted_offpk) * 100

ax.plot(exc_pk, sorted_peak, color=RED, linewidth=2, label='Peak demand periods\n(winter weekdays 16–20h UTC)')
ax.plot(exc_op, sorted_offpk, color=BLUE, linewidth=2, alpha=0.7, label='Off-peak periods')
ax.axhline(peak_pcts[10], color=RED, linestyle=':', alpha=0.8)
ax.text(95, peak_pcts[10] + 100, f'Peak P10 = {peak_pcts[10]:.0f}MW', color=RED, fontsize=9)
ax.set_xlabel('% of periods where wind exceeds this level')
ax.set_ylabel('Wind Generation (MW)')
ax.set_title('Reliability Curve: Peak vs Off-peak Periods')
ax.legend(fontsize=10)
ax.set_xlim(0, 100)
ax.set_facecolor('#1e293b')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('peak_vs_offpeak.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Temporal Variability: Rolling 24h Minimum

Grid planners need to understand **sustained low-wind events** — not just snapshot lows, but prolonged periods where wind stays low. We compute the rolling 24h minimum generation.

In [ ]:
# Set datetime index for rolling operations
ts = actuals.set_index('target_time')['actual_mw'].sort_index()

# Rolling windows (48 = 24h at 30-min resolution)
roll_min_24h  = ts.rolling(window=48, min_periods=24).min()
roll_mean_24h = ts.rolling(window=48, min_periods=24).mean()

# Worst sustained 24h-min events
worst_days = roll_min_24h.nsmallest(10)
print('Top 10 lowest 24h rolling minimums (MW):')
for t, v in worst_days.items():
    print(f'  {t}  →  {v:.0f} MW')

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
fig.suptitle('UK Wind Generation Time Series (Jan–Mar 2025)', color='#f1f5f9', fontsize=14)

ax = axes[0]
ax.fill_between(ts.index, ts.values, alpha=0.4, color=BLUE)
ax.plot(ts.index, ts.values, color=BLUE, linewidth=0.6, alpha=0.8)
ax.plot(roll_mean_24h.index, roll_mean_24h.values, color=AMBER, linewidth=1.5, label='24h rolling mean')
ax.set_ylabel('Generation (MW)')
ax.set_title('Actual Wind Generation')
ax.legend(fontsize=9)
ax.set_facecolor('#1e293b')

ax = axes[1]
ax.fill_between(roll_min_24h.index, roll_min_24h.values, alpha=0.4, color=RED)
ax.plot(roll_min_24h.index, roll_min_24h.values, color=RED, linewidth=1, label='24h rolling minimum')
p10_24h = np.percentile(roll_min_24h.dropna(), 10)
ax.axhline(p10_24h, color=AMBER, linestyle='--', linewidth=1.5)
ax.text(roll_min_24h.index[-1], p10_24h + 50,
        f'P10 of 24h rolling min\n= {p10_24h:.0f} MW', color=AMBER, fontsize=8, ha='right')
ax.set_ylabel('Generation (MW)')
ax.set_title('24h Rolling Minimum (worst window each day)')
ax.legend(fontsize=9)
ax.set_facecolor('#1e293b')

plt.tight_layout()
plt.savefig('rolling_min.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Monthly & Weekly Patterns

In [ ]:
monthly = (
    actuals.groupby('month_name')['actual_mw']
    .agg(['mean', 'median',
          lambda x: x.quantile(0.1),
          lambda x: x.quantile(0.9),
          'min', 'max'])
    .reset_index()
)
monthly.columns = ['month', 'mean', 'median', 'p10', 'p90', 'min', 'max']

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(monthly))
ax.bar(x, monthly['p90'] - monthly['p10'],
       bottom=monthly['p10'], color=BLUE, alpha=0.3, label='P10–P90 range')
ax.plot(x, monthly['median'], 'o-', color=GREEN, linewidth=2, markersize=6, label='Median')
ax.plot(x, monthly['mean'], 's--', color=AMBER, linewidth=1.5, markersize=5, label='Mean')
ax.plot(x, monthly['p10'], '^:', color=RED, linewidth=1.5, markersize=5, label='P10 (reliable floor)')
ax.set_xticks(list(x))
ax.set_xticklabels(monthly['month'])
ax.set_ylabel('Wind Generation (MW)')
ax.set_title('Monthly Wind Generation Statistics')
ax.legend(fontsize=9)
ax.set_facecolor('#1e293b')
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('monthly_stats.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Capacity Factor Analysis

The **capacity factor** tells us what fraction of installed capacity is actually being used. The UK installed onshore + offshore wind capacity was approximately **30–32 GW** by end of 2024.

We use this to contextualise our percentile results.

In [ ]:
# Approximate installed capacity (end 2024, onshore + offshore)
# Source: DESNZ / NESO, approximately 30-32 GW
INSTALLED_CAPACITY_GW = 31  # conservative estimate

actuals['capacity_factor'] = actuals['actual_mw'] / (INSTALLED_CAPACITY_GW * 1000)

cf_pcts = {p: np.percentile(actuals['capacity_factor'], p) for p in pcts}
print(f'Assuming installed capacity: {INSTALLED_CAPACITY_GW} GW')
print('\nCapacity Factor Percentiles:')
for p, v in cf_pcts.items():
    mw = v * INSTALLED_CAPACITY_GW * 1000
    print(f'  P{p:>2}: CF = {v:.3f}  ({mw:>7,.0f} MW)')

print(f'\nMean CF: {actuals.capacity_factor.mean():.3f} ({actuals.capacity_factor.mean()*100:.1f}%)')

## 7. Recommendation: Reliable Wind Capacity

### Summary Table

In [ ]:
# Build recommendation summary
data_for_table = [
    ('Overall P10 (90% reliable)', percentile_vals[10], 'Conservative baseline — wind exceeds this 90% of all half-hour periods'),
    ('Peak-period P10',            peak_pcts[10],       'Conservative during winter weekday evenings — highest reliability demand'),
    ('Overall P25 (75% reliable)', percentile_vals[25], 'Moderate reliability — appropriate for non-critical planning'),
    ('Overall Median (P50)',       percentile_vals[50], 'Central estimate — expected output in an average period'),
    ('Overall Mean',               actuals.actual_mw.mean(), 'Average output including high/low extremes'),
]

print('='*80)
print('RELIABLE WIND CAPACITY RECOMMENDATION SUMMARY')
print('='*80)
print(f'{"Metric":<35} {"MW":>8}  Description')
print('-'*80)
for name, mw, desc in data_for_table:
    print(f'{name:<35} {mw:>8,.0f}  {desc}')
print('='*80)

print()
print('RECOMMENDATION:')
print(f'''\nFor operational planning purposes — where reliability matters — we recommend using
the PEAK-PERIOD P10 figure of ~{peak_pcts[10]:,.0f} MW as the "firm" reliable wind contribution.

Rationale:
  1. It represents the output level exceeded during 90% of peak demand periods (winter
     weekday evenings), when grid reliability matters most.
  2. It accounts for the correlation between high demand and potentially low wind
     (cold dark doldrums — high pressure, low wind, cold temperatures).
  3. For capacity adequacy assessments (Ofgem/NESO framework), this conservative
     approach avoids over-crediting variable renewables.

For energy (not capacity) planning, the mean figure of ~{actuals.actual_mw.mean():,.0f} MW
is appropriate — representing the expected contribution to annual energy output.

Caveats:
  - This analysis covers only Jan–Mar 2025. A full-year dataset is needed to capture
    summer/autumn wind patterns (which may be lower).
  - As UK installed wind capacity grows, absolute reliable output will increase
    proportionally (assuming capacity factor is maintained).
  - Interconnector availability and storage can mitigate low-wind periods,
    potentially allowing a higher reliable figure in practice.
''')

In [ ]:
# Final summary visualisation
fig, ax = plt.subplots(figsize=(12, 6))

# Exceedance curve — all
sorted_all = np.sort(actuals['actual_mw'].values)[::-1]
exc_all = np.arange(1, len(sorted_all)+1) / len(sorted_all) * 100
ax.plot(exc_all, sorted_all, color=BLUE, linewidth=2.5, label='All periods', zorder=3)

# Exceedance curve — peak
sorted_pk = np.sort(peak['actual_mw'].values)[::-1]
exc_pk = np.arange(1, len(sorted_pk)+1) / len(sorted_pk) * 100
ax.plot(exc_pk, sorted_pk, color=RED, linewidth=2, linestyle='--', label='Peak demand periods', zorder=3)

# Recommendation annotations
recommendations = [
    (90, peak_pcts[10],       RED,   'RECOMMENDED\nPeak P10'),
    (90, percentile_vals[10], BLUE,  'Overall P10'),
    (50, percentile_vals[50], AMBER, 'Median (P50)'),
]
for exc_val, mw, col, label in recommendations:
    ax.plot(exc_val, mw, 'o', color=col, markersize=8, zorder=5)
    ax.annotate(f'{label}\n{mw:,.0f} MW',
                xy=(exc_val, mw), xytext=(exc_val - 12, mw + 500),
                color=col, fontsize=8.5, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=col, lw=1))

ax.fill_between(exc_all,
                np.interp(exc_all, exc_pk[::-1], sorted_pk[::-1]),
                sorted_all, alpha=0.08, color=PURPLE, label='Peak deficit vs average')

ax.set_xlabel('% of time when wind generation exceeds this level', fontsize=12)
ax.set_ylabel('Wind Generation (MW)', fontsize=12)
ax.set_title('UK Wind Reliability Curve: All Periods vs Peak Demand Periods\nJan–Mar 2025', fontsize=13)
ax.set_xlim(0, 100)
ax.set_ylim(0)
ax.legend(fontsize=10)
ax.set_facecolor('#1e293b')
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('reliability_recommendation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reliability_recommendation.png')

## 8. Conclusions

### How much wind can we reliably expect?

| Reliability level | MW estimate | When to use |
|---|---|---|
| **Firm (P10 at peak)** | ~X,XXX MW | Capacity adequacy, grid planning |
| **Conservative (P10 overall)** | ~X,XXX MW | Cautious operational planning |
| **Moderate (P25 overall)** | ~X,XXX MW | Probabilistic forecasting |
| **Average (mean)** | ~X,XXX MW | Annual energy planning |

*(Exact figures populated from analysis above)*

### Analytical reasoning

**Why not use the minimum?** The minimum ever recorded (near-zero) would render wind's contribution negligible — this is overly conservative and doesn't reflect how grids are actually planned. Grids always carry reserve margins and interconnector capacity.

**Why not use the mean?** The mean (~8–10 GW for UK wind) is useful for energy accounting but is dangerous for reliability planning — half the time, output is *below* the mean, and in a cold dark doldrum event output could be well below this for days at a time.

**Why P10 at peak?** This is conceptually aligned with the **Equivalent Firm Capacity (EFC)** methodology used by NESO (formerly National Grid ESO) for variable renewable capacity credit. It represents what wind can *reliably* contribute when it matters most — during high-demand periods — without over-crediting.

**Seasonal caveat**: Winter (Jan–Mar) data used here may overestimate reliable output vs summer months, which are typically lower wind periods in the UK. A full-year analysis is recommended for final planning figures.

## 1. Load and Prepare Historical Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime, timedelta

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Fetch actual generation data
BMRS_BASE = 'https://data.elexon.co.uk/bmrs/api/v1'
DATA_START = '2025-01-01'
DATA_END = '2025-03-18'

def fetch_actuals(from_date, to_date):
    """Fetch actual wind generation from FUELHH"""
    url = f"{BMRS_BASE}/datasets/FUELHH/stream?from={from_date}&to={to_date}&fuelType=WIND"
    print(f"Fetching historical actuals from {from_date} to {to_date}...")
    try:
        response = requests.get(url)
        if response.ok:
            data = response.json()
            df = pd.json_normalize(data)
            print(f"Fetched {len(df)} records")
            return df
        else:
            print(f"Error: {response.status_code}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

# Load data
actuals_df = fetch_actuals(DATA_START, DATA_END)

if not actuals_df.empty:
    # Clean data
    actuals_df['startTime'] = pd.to_datetime(actuals_df['startTime'])
    actuals_df['generation'] = pd.to_numeric(actuals_df['generation'], errors='coerce')
    
    # Remove NaNs
    actuals_df = actuals_df.dropna(subset=['generation'])
    
    # Sort by time
    actuals_df = actuals_df.sort_values('startTime')
    
    print(f"\nData Summary:")
    print(f"  Period: {actuals_df['startTime'].min()} to {actuals_df['startTime'].max()}")
    print(f"  Total records: {len(actuals_df)}")
    print(f"  Data points per day: {len(actuals_df) / (actuals_df['startTime'].max() - actuals_df['startTime'].min()).days:.0f}")

## 2. Statistical Analysis of Generation Distribution

In [ ]:
# Get maximum installed capacity from the data
max_generation = actuals_df['generation'].max()
p95_generation = actuals_df['generation'].quantile(0.95)
p90_generation = actuals_df['generation'].quantile(0.90)
p75_generation = actuals_df['generation'].quantile(0.75)
p50_generation = actuals_df['generation'].quantile(0.50)
p25_generation = actuals_df['generation'].quantile(0.25)
p10_generation = actuals_df['generation'].quantile(0.10)

print("="*70)
print("GENERATION STATISTICS & PERCENTILES")
print("="*70)
print(f"\nGeneration Range:")
print(f"  Maximum observed: {max_generation:,.0f} MW (100th percentile)")
print(f"  95th percentile: {p95_generation:,.0f} MW (high generation)")
print(f"  90th percentile: {p90_generation:,.0f} MW")
print(f"  75th percentile: {p75_generation:,.0f} MW (good generation)")
print(f"  Median (50th):   {p50_generation:,.0f} MW (typical)")
print(f"  25th percentile: {p25_generation:,.0f} MW")
print(f"  10th percentile: {p10_generation:,.0f} MW (low generation)")
print(f"  Minimum:        {actuals_df['generation'].min():,.0f} MW")

# Capacity factors
print(f"\nCapacity Metrics:")
print(f"  Average generation: {actuals_df['generation'].mean():,.0f} MW")
print(f"  Std deviation: {actuals_df['generation'].std():,.0f} MW")
print(f"  Coefficient of variation: {(actuals_df['generation'].std() / actuals_df['generation'].mean()):.2f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(actuals_df['generation'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(actuals_df['generation'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 0].axvline(actuals_df['generation'].median(), color='orange', linestyle='--', linewidth=2, label='Median')
axes[0, 0].set_title('Distribution of Wind Generation', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Generation (MW)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Time series
axes[0, 1].plot(actuals_df['startTime'], actuals_df['generation'], linewidth=0.5, color='steelblue', alpha=0.7)
axes[0, 1].set_title('Wind Generation Over Time', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Generation (MW)')
axes[0, 1].grid(True, alpha=0.3)

# Percentile chart
percentiles = [10, 25, 50, 75, 90, 95]
values = [actuals_df['generation'].quantile(p/100) for p in percentiles]
axes[1, 0].bar([f'P{p}' for p in percentiles], values, color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Generation Percentiles', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Generation (MW)')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Cumulative distribution
sorted_gen = np.sort(actuals_df['generation'])
axes[1, 1].plot(sorted_gen, np.arange(1, len(sorted_gen)+1)/len(sorted_gen)*100, linewidth=2, color='steelblue')
axes[1, 1].set_title('Cumulative Distribution of Generation', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Generation (MW)')
axes[1, 1].set_ylabel('Cumulative Probability (%)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Reliability During Peak Demand Periods

In [ ]:
# Analyze generation by hour (UK peak demand: 17:00-20:00 GMT)
actuals_df['hour'] = actuals_df['startTime'].dt.hour
actuals_df['month'] = actuals_df['startTime'].dt.month
actuals_df['is_winter'] = actuals_df['month'].isin([11, 12, 1, 2])  # Nov-Feb
actuals_df['is_peak_demand'] = actuals_df['hour'].isin([17, 18, 19])  # 5-8 PM

hourly_stats = actuals_df.groupby('hour')['generation'].agg(['mean', 'median', lambda x: x.quantile(0.10), lambda x: x.quantile(0.50), lambda x: x.quantile(0.90)]).round(0)
hourly_stats.columns = ['Mean', 'Median', 'P10', 'P50', 'P90']

peak_demand_data = actuals_df[actuals_df['is_peak_demand']]
winter_peak_data = actuals_df[actuals_df['is_winter'] & actuals_df['is_peak_demand']]

print("\n" + "="*70)
print("PEAK DEMAND PERIOD ANALYSIS (17:00-19:59 UTC)")
print("="*70)
print(f"\nPeak demand hours (17-19:00 UTC):")
print(f"  Average generation: {peak_demand_data['generation'].mean():,.0f} MW")
print(f"  Median generation: {peak_demand_data['generation'].median():,.0f} MW")
print(f"  P10 (low availability): {peak_demand_data['generation'].quantile(0.10):,.0f} MW")
print(f"  P90 (high availability): {peak_demand_data['generation'].quantile(0.90):,.0f} MW")

print(f"\nWinter peak demand (Nov-Feb, 17-19:00 UTC):")
print(f"  Average generation: {winter_peak_data['generation'].mean():,.0f} MW")
print(f"  Median generation: {winter_peak_data['generation'].median():,.0f} MW")
print(f"  P10 (low availability): {winter_peak_data['generation'].quantile(0.10):,.0f} MW")
print(f"  P90 (high availability): {winter_peak_data['generation'].quantile(0.90):,.0f} MW")
print(f"  Sample size: {len(winter_peak_data)} half-hourly observations")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Generation by hour
axes[0].plot(hourly_stats.index, hourly_stats['Mean'], 'o-', linewidth=2, label='Mean', color='steelblue')
axes[0].fill_between(hourly_stats.index, hourly_stats['P10'], hourly_stats['P90'], alpha=0.2, color='steelblue', label='P10-P90')
axes[0].axvspan(17, 20, alpha=0.1, color='red', label='Peak demand hours')
axes[0].set_title('Average Generation by Hour of Day', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hour of Day (UTC)')
axes[0].set_ylabel('Generation (MW)')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Peak demand comparison
peak_stats = pd.DataFrame({
    'All Hours': [actuals_df['generation'].quantile(0.10), actuals_df['generation'].median(), actuals_df['generation'].quantile(0.90)],
    'Peak Demand': [peak_demand_data['generation'].quantile(0.10), peak_demand_data['generation'].median(), peak_demand_data['generation'].quantile(0.90)],
    'Winter Peak': [winter_peak_data['generation'].quantile(0.10), winter_peak_data['generation'].median(), winter_peak_data['generation'].quantile(0.90)]
}, index=['P10', 'Median', 'P90'])

peak_stats.plot(kind='bar', ax=axes[1], color=['steelblue', 'orange', 'red'], alpha=0.7)
axes[1].set_title('Generation Percentiles: All Hours vs Peak Demand', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Generation (MW)')
axes[1].set_xlabel('Percentile')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3, axis='y')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 4. Reliable Capacity Recommendation

In [ ]:
print("="*70)
print("RELIABLE CAPACITY RECOMMENDATIONS FOR UK GRID PLANNING")
print("="*70)

# Define reliability tiers
p10_overall = actuals_df['generation'].quantile(0.10)
p25_overall = actuals_df['generation'].quantile(0.25)
p50_overall = actuals_df['generation'].quantile(0.50)
p10_winter_peak = winter_peak_data['generation'].quantile(0.10)

print("\n### RECOMMENDATION FRAMEWORK ###\n")

print("For BASELOAD/GUARANTEED supply (90% confidence):")
print(f"  Reliable capacity: {p10_overall:,.0f} MW")
print(f"  Rationale: P10 (10th percentile) - generation below this 10% of time")
print(f"  Use case: Contracts with 90% certainty for peak demand coverage")

print("\nFor INTERMEDIATE supply (75% confidence):")
print(f"  Reliable capacity: {p25_overall:,.0f} MW")
print(f"  Rationale: P25 (25th percentile) - generation below 25% of time")
print(f"  Use case: Planning for 75% guaranteed availability")

print("\nFor TYPICAL  supply (50% confidence):")
print(f"  Reliable capacity: {p50_overall:,.0f} MW")
print(f"  Rationale: Median generation - present 50% of the time")
print(f"  Use case: Typical grid planning assumption")

print("\n### WINTER PEAK DEMAND ASSESSMENT ###\n")
print(f"During winter peak hours (Nov-Feb, 17-19 UTC):")
print(f"  90% reliable capacity: {winter_peak_data['generation'].quantile(0.10):,.0f} MW")
print(f"  75% reliable capacity: {winter_peak_data['generation'].quantile(0.25):,.0f} MW")
print(f"  Median capacity: {winter_peak_data['generation'].median():,.0f} MW")

print("\n### FINAL RECOMMENDATION ###\n")

# Develop tiered recommendation
rec_baseload = int(p10_overall / 100) * 100  # Round down to nearest 100
rec_intermediate = int(p25_overall / 100) * 100
rec_typical = int(p50_overall / 100) * 100

print(f"Based on first-principles analysis of {len(actuals_df):,} half-hourly observations:")
print(f"\n✓ CONSERVATIVE (90% confidence): {rec_baseload:,} MW")
print(f"  - Suitable for critical grid baseload requirements")
print(f"  - Guarantees availability 90% of operational hours")
print(f"  - Accounts for low-wind periods (worst 10%)")

print(f"\n✓ MODERATE (75% confidence): {rec_intermediate:,} MW")
print(f"  - Balanced approach for medium-term grid planning")
print(f"  - Achieves 75% availability threshold")
print(f"  - Most realistic for grid integration targets")

print(f"\n✓ OPTIMISTIC (50% confidence): {rec_typical:,} MW")
print(f"  - Maximum generation under typical conditions")
print(f"  - Appropriate for supplementary capacity")
print(f"  - Does not account for drought periods")

print("\n### KEY INSIGHTS ###\n")
print(f"1. Weather dependency: High variance (σ={actuals_df['generation'].std():,.0f} MW)")
print(f"   - Wind is highly variable; reliable capacity << installed capacity")
print(f"   - Capacity credit lower than conventional generators (30-40% vs 90%)")

print(f"\n2. Seasonal patterns:")
print(f"   - Winter advantage for peak demand: {winter_peak_data['generation'].mean():,.0f} MW avg winter peak")
print(f"   - vs {peak_demand_data['generation'].mean():,.0f} MW all-hours average")

print(f"\n3. Correlation with demand:")
print(f"   - Wind availability during peak demand (17-19 UTC): {(peak_demand_data['generation'].mean()/actuals_df['generation'].mean()):.1%}")
print(f"   - Moderate overlap with peak demand periods")

print("\n" + "="*70)